# Plotly: exploring data with interactive charts

A reader wants to explore patterns and inspect individual observations without generating a new chart each time. Build an interactive report that reveals useful detail while keeping summaries, filters and hidden traces understandable. Interactivity should answer a question, not obscure which data is currently displayed.

A static chart shows one view. An interactive chart lets a reader inspect a point, hide a series, zoom into a period, or switch between related views. Plotly creates these charts in a browser and can save them as standalone HTML files.

We will explore **invented café sales data**, then build an interactive report. The data does not describe a real business. All examples run offline after installation, and the final HTML embeds the JavaScript needed for interaction.

Run cells from top to bottom. Familiarity with Pandas tables and basic chart selection is helpful.

## 1. Install and import

If needed, run `%pip install plotly pandas numpy nbformat` in a separate cell.

| Interface | Use |
| --- | --- |
| `plotly.express` (`px`) | Build common charts directly from a table |
| `plotly.graph_objects` (`go`) | Construct and customize individual traces |
| `make_subplots` | Arrange multiple charts within a figure |

Both Express and graph objects produce a Figure. Its `data` contains traces (such as one branch's line), and its `layout` contains titles, axes, legends, and controls. `show()` uses your environment's renderer; the HTML export also works when notebook rendering is unavailable.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"
print("Plotly:", plotly.__version__)

## 2. Create a tidy table

Each row represents one branch on one day. Orders are a count, revenue is in dollars, and average order value is revenue divided by orders. A seeded generator makes the example reproducible.

Long-form data repeats the branch label on each row, which makes it easy to map that column to color or facets.

In [ ]:
rng = np.random.default_rng(42)
branches = ["Central", "Riverside", "Uptown"]
colors = {"Central": "#0072B2", "Riverside": "#D55E00", "Uptown": "#009E73"}
dates = pd.date_range("2025-01-01", periods=60, freq="D")
rows = []
for index, branch in enumerate(branches):
    for day_index, date in enumerate(dates):
        weekend = date.dayofweek >= 5
        orders = max(10, int(rng.normal(75 + 10 * index + 18 * weekend + 0.3 * day_index, 8)))
        average_order = max(5, rng.normal(12 + index, 1))
        rows.append({"date": date, "branch": branch, "orders": orders,
                     "revenue": round(orders * average_order, 2)})
sales = pd.DataFrame(rows).sort_values(["branch", "date"]).reset_index(drop=True)
sales["average_order"] = sales["revenue"] / sales["orders"]
print(sales.head())
print("Rows:", len(sales), "Missing values:", sales.isna().sum().to_dict())

## 3. Start with an interactive line chart

Line charts connect observations in their supplied order, so sort time values first. Color identifies the branch consistently throughout this notebook.

Hover to inspect values, click a legend item to hide or show a branch, and use the toolbar to zoom or reset the view. `hovermode="x unified"` collects values at a shared date in one hover box.

In [ ]:
line_fig = px.line(sales, x="date", y="revenue", color="branch",
                   color_discrete_map=colors, category_orders={"branch": branches},
                   labels={"date": "Date", "revenue": "Daily revenue ($)", "branch": "Branch"},
                   title="Daily café revenue — synthetic data")
line_fig.update_layout(hovermode="x unified")
line_fig.show()

## 4. Give a surprising point enough context to investigate

Choose which fields would explain an unusual point before adding hover data. Hover over one point and trace it to a source row. Labels should make the observation interpretable without exposing irrelevant fields.

A scatter point below is one branch-day. Hover details include the date, orders, and revenue. `hover_data` can format numeric fields or suppress redundant fields; axis labels still explain the chart without hovering.

Larger order counts tend to accompany larger revenue because revenue was generated from count times average order value. This pattern follows the data construction.

In [ ]:
scatter_fig = px.scatter(sales, x="orders", y="revenue", color="branch", symbol="branch",
                         color_discrete_map=colors, hover_name="branch",
                         hover_data={"date": "|%b %d, %Y", "revenue": ":.2f", "average_order": ":.2f", "branch": False},
                         labels={"orders": "Daily orders", "revenue": "Daily revenue ($)",
                                 "average_order": "Average order ($)"},
                         title="Orders and revenue for each branch-day")
scatter_fig.update_traces(marker={"size": 8, "opacity": 0.7})
scatter_fig.show()

## 5. Summarize once before comparing categories

Predict one total from the source rows before drawing the bar chart. Verify that the chart displays the intended aggregate rather than multiple overlapping marks that only look like a summary.

`px.bar` draws the rows you give it; it does not automatically calculate the business summary you intended. Group explicitly to obtain one row per branch.

For average order value across the period, divide total revenue by total orders. Averaging daily averages would weight quiet and busy days equally.

In [ ]:
branch_summary = sales.groupby("branch", as_index=False).agg(
    revenue=("revenue", "sum"), orders=("orders", "sum")
)
branch_summary["average_order"] = branch_summary["revenue"] / branch_summary["orders"]
bar_fig = px.bar(branch_summary, x="branch", y="revenue", color="branch", color_discrete_map=colors,
                 category_orders={"branch": branches}, text="revenue",
                 labels={"branch": "Branch", "revenue": "Total revenue ($)"}, title="Revenue across all 60 days")
bar_fig.update_traces(texttemplate="$%{y:,.0f}", textposition="outside")
bar_fig.update_layout(showlegend=False)
bar_fig.update_yaxes(rangemode="tozero")
bar_fig.show()
print(branch_summary.round(2))

## 6. Inspect distributions

Histograms group numerical values into intervals. A box plot summarizes the median, quartiles, and spread; adding all observations helps reveal what the summary represents.

`nbins` is a requested bin count and may be adjusted by Plotly's binning rules. Counts can reflect both distribution shape and sample size; this example has the same 60 days per branch.

In [ ]:
hist_fig = px.histogram(sales, x="revenue", color="branch", color_discrete_map=colors,
                        nbins=15, barmode="overlay", opacity=0.5,
                        labels={"revenue": "Daily revenue ($)"}, title="Distribution of daily revenue")
hist_fig.update_yaxes(title_text="Number of branch-days")
hist_fig.show()
box_fig = px.box(sales, x="branch", y="revenue", color="branch", color_discrete_map=colors,
                 points="all", category_orders={"branch": branches},
                 labels={"branch": "Branch", "revenue": "Daily revenue ($)"}, title="Spread within each branch")
box_fig.update_layout(showlegend=False)
box_fig.show()

## 7. Facets compare subsets

A facet creates one panel for each value of a category. Shared scales allow direct comparisons of levels and variation. Independent scales may reveal within-group detail, but they make visual height less comparable.

Here all panels share the same revenue scale, so the lines can be compared across branches.

In [ ]:
facet_fig = px.line(sales, x="date", y="revenue", facet_col="branch", color="branch",
                    color_discrete_map=colors, category_orders={"branch": branches},
                    labels={"date": "Date", "revenue": "Revenue ($)"},
                    title="Daily revenue with one panel per branch")
facet_fig.update_layout(showlegend=False)
facet_fig.show()

## 8. Heatmaps compare a matrix

Summarize mean daily revenue by weekday and branch. This is a mean, not a total: the 60-day window contains unequal numbers of some weekdays.

Use an explicit weekday order so the chart follows the calendar. The colorbar states the quantity encoded by color.

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sales["weekday"] = sales["date"].dt.day_name()
weekday_matrix = sales.pivot_table(index="branch", columns="weekday", values="revenue", aggfunc="mean")
weekday_matrix = weekday_matrix.reindex(index=branches, columns=weekday_order)
heatmap_fig = go.Figure(go.Heatmap(z=weekday_matrix.to_numpy(), x=weekday_order, y=branches,
    colorscale="Viridis", colorbar={"title": "Mean revenue ($)"},
    hovertemplate="%{y}<br>%{x}<br>Mean revenue: $%{z:,.2f}<extra></extra>"))
heatmap_fig.update_layout(title="Mean daily revenue by weekday", xaxis_title="Weekday", yaxis_title="Branch")
heatmap_fig.show()

## 9. Use graph objects for explicit trace control

Each `go.Scatter` below defines one branch's line. `mode="lines+markers"` controls its marks. The hover template defines the text precisely, and `<extra></extra>` suppresses the secondary trace-name box.

Graph objects are useful when a chart needs trace-by-trace construction. Express figures can also be customized with `update_traces` and `update_layout`.

In [ ]:
custom_fig = go.Figure()
for branch in branches:
    subset = sales.loc[sales["branch"] == branch]
    custom_fig.add_trace(go.Scatter(x=subset["date"], y=subset["orders"], name=branch,
        mode="lines+markers", line={"color": colors[branch]},
        hovertemplate="%{x|%b %d}<br>Orders: %{y}<extra></extra>"))
custom_fig.update_layout(title="Daily orders", xaxis_title="Date", yaxis_title="Orders", hovermode="x unified")
custom_fig.show()

## 10. Add a range slider for time exploration

A range slider provides an overview while narrowing the displayed dates. Range-selector buttons offer convenient presets.

We copy the original Figure so this example can customize its controls independently. These controls change the displayed range; they do not recalculate or download data.

In [ ]:
range_fig = go.Figure(line_fig)
range_fig.update_xaxes(rangeslider_visible=True, rangeselector={"buttons": [
    {"count": 14, "label": "2 weeks", "step": "day", "stepmode": "backward"},
    {"count": 1, "label": "1 month", "step": "month", "stepmode": "backward"},
    {"label": "All", "step": "all"},
]})
range_fig.update_layout(title="Explore a shorter revenue window", height=550)
range_fig.show()

## 11. Make each dropdown state mean what its label says

Predict which traces each selection should reveal. Exercise every selection and inspect legends, titles and units. A dropdown can create a misleading comparison if a title or axis retains meaning from a previous state.

The visibility list has one Boolean per trace, in trace order. The All button shows three branches; each other choice shows one. `method="update"` changes trace properties and layout together.

All traces are already embedded in the figure, so the menu works in exported HTML without a Python process. It is not a server query or a live data refresh.

In [ ]:
dropdown_fig = go.Figure(line_fig)
buttons = [{"label": "All branches", "method": "update", "args": [
    {"visible": [True] * len(branches)}, {"title": "Daily revenue — all branches"}]}]
for branch in branches:
    visible = [trace.name == branch for trace in dropdown_fig.data]
    buttons.append({"label": branch, "method": "update", "args": [
        {"visible": visible}, {"title": f"Daily revenue — {branch}"}]})
dropdown_fig.update_layout(title="Daily revenue — all branches",
    updatemenus=[{"buttons": buttons, "direction": "down", "x": 0, "y": 1.18}], margin={"t": 110})
dropdown_fig.show()

## 12. Show missing data as a gap

A missing measurement is not a zero. Set a few values to `NaN` and keep `connectgaps=False` so the line does not bridge unobserved values.

Labels or annotations can explain a known data gap. The values here are deliberately removed to demonstrate the behavior.

In [ ]:
gap_data = sales.loc[sales["branch"] == "Central"].copy()
gap_data.loc[gap_data["date"].between("2025-01-20", "2025-01-23"), "revenue"] = np.nan
gap_fig = px.line(gap_data, x="date", y="revenue", markers=True,
                  labels={"date": "Date", "revenue": "Revenue ($)"}, title="Four deliberately missing days")
gap_fig.update_traces(connectgaps=False)
gap_fig.add_annotation(x="2025-01-21", y=float(gap_data["revenue"].median()),
                       text="Missing readings", showarrow=False)
gap_fig.show()

## 13. Explore a bundled real dataset

Plotly Express includes the historical Gapminder sample. Here we select 2007, the last year in that sample; this is not current country data. GDP per capita is purchasing-power-adjusted income in the dataset's units, and life expectancy is in years.

Bubble area represents population, and the GDP axis is logarithmic. Relationships between countries do not establish individual-level relationships or causation. The dataset is documented by [Gapminder](https://www.gapminder.org/data/).

In [ ]:
gapminder = px.data.gapminder()
country_data = gapminder.loc[gapminder["year"] == 2007]
country_fig = px.scatter(country_data, x="gdpPercap", y="lifeExp", size="pop", color="continent",
                         hover_name="country", log_x=True, size_max=45,
                         labels={"gdpPercap": "GDP per capita (PPP, dataset units)", "lifeExp": "Life expectancy (years)",
                                 "pop": "Population", "continent": "Continent"},
                         title="Gapminder: countries in 2007")
country_fig.show()

## 14. Complete exercise: assemble an interactive report

Combine revenue over time with total revenue by branch. The two panels answer different questions: variation through the period and cumulative contribution.

Matching legend groups connect each branch's traces across panels. The dropdown visibility masks cover every trace, including the bars. Totals remain for the complete 60-day dataset even when zooming the time panel.

In [ ]:
report = make_subplots(rows=2, cols=1, vertical_spacing=0.18,
                       subplot_titles=("Daily revenue", "Total revenue across all 60 days"))
for branch in branches:
    subset = sales.loc[sales["branch"] == branch]
    report.add_trace(go.Scatter(x=subset["date"], y=subset["revenue"], name=branch,
        legendgroup=branch, mode="lines", line={"color": colors[branch]},
        hovertemplate="%{x|%b %d}<br>Revenue: $%{y:,.2f}<extra></extra>"), row=1, col=1)
for branch in branches:
    total = float(branch_summary.loc[branch_summary["branch"] == branch, "revenue"].iloc[0])
    report.add_trace(go.Bar(x=[branch], y=[total], name=branch, legendgroup=branch,
        showlegend=False, marker_color=colors[branch],
        hovertemplate="%{x}<br>Total: $%{y:,.2f}<extra></extra>"), row=2, col=1)
report_buttons = [{"label": "All branches", "method": "update", "args": [
    {"visible": [True] * len(report.data)}, {"title": "Café report — all branches (synthetic data)"}]}]
for branch in branches:
    report_buttons.append({"label": branch, "method": "update", "args": [
        {"visible": [trace.name == branch for trace in report.data]},
        {"title": f"Café report — {branch} (synthetic data)"}]})
report.update_layout(title="Café report — all branches (synthetic data)", height=800,
    updatemenus=[{"buttons": report_buttons, "x": 0, "y": 1.13}], margin={"t": 130},
    legend={"groupclick": "togglegroup"})
report.update_xaxes(title_text="Date", row=1, col=1)
report.update_yaxes(title_text="Daily revenue ($)", row=1, col=1)
report.update_xaxes(title_text="Branch", categoryorder="array", categoryarray=branches, row=2, col=1)
report.update_yaxes(title_text="Total revenue ($)", rangemode="tozero", row=2, col=1)
report.show()

## 15. Verify that the report works outside the notebook

Open the saved HTML independently. Check hover behavior and controls, and verify whether the export includes its JavaScript dependencies. A notebook preview alone does not show what the recipient will receive.

`include_plotlyjs=True` embeds the rendering library so this report works offline in a browser. Using `"cdn"` instead would reduce file size but require access to an external library. The HTML contains the chart's data, including traces hidden by controls.

We also save the figure as JSON for reloading in Python and the branch totals as CSV. Running this cell creates `outputs` and replaces these named files. No server or account is required to open the HTML file.

In [ ]:
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
html_path = output_dir / "plotly_cafe_report.html"
json_path = output_dir / "plotly_cafe_report.json"
csv_path = output_dir / "plotly_branch_totals.csv"
report.write_html(html_path, include_plotlyjs=True, full_html=True, auto_open=False,
                  config={"responsive": True, "displaylogo": False})
report.write_json(json_path)
branch_summary.to_csv(csv_path, index=False)
reloaded_report = pio.read_json(json_path)
for path in [html_path, json_path, csv_path]:
    print(f"Saved: {path} ({path.stat().st_size:,} bytes)")
print("Reloaded traces:", len(reloaded_report.data))

## 16. Practice and troubleshooting

1. Add order-count charts and label their units clearly.
2. Change the report to weekly totals. Decide how to label partial weeks.
3. Add a dropdown choice that compares only Central and Riverside; ensure it updates both panels.
4. Rebuild the heatmap using daily order counts and explain the difference from revenue.

| Symptom | What to check |
| --- | --- |
| Chart is blank in a notebook | Check the renderer or open the exported HTML in a browser |
| Time line zigzags unexpectedly | Sort rows by date within each series |
| Bars show unintended stacking | Aggregate to the intended grain and check `barmode` |
| Menu hides the wrong traces | Match visibility-mask length and order to `fig.data` |
| HTML fails offline | Embed Plotly.js with `include_plotlyjs=True` |
| Export is large | Embedded JavaScript and data both contribute to size |
| Zoom does not update totals | Axis controls change the view; custom recalculation requires additional logic |

Static image export is a separate workflow with additional dependencies; this notebook exports HTML to preserve interactivity.

## Further reading

See the official [Plotly Express guide](https://plotly.com/python/plotly-express/), [hover formatting](https://plotly.com/python/hover-text-and-formatting/), [dropdown controls](https://plotly.com/python/dropdowns/), [renderers](https://plotly.com/python/renderers/), and [interactive HTML export](https://plotly.com/python/interactive-html-export/) documentation.

## Explain what the evidence supports

Identify an insight that requires inspecting a point or changing a control. Describe which rows are visible in that state and verify that the exported report supports the same investigation.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: ship a latency dashboard within a browser-size budget

A service release appears slower in an overall latency dashboard, yet engineers say each route improved. Reconstruct the report from 2,000 generated request events with a changed traffic mix, missing timeout measurements, three malformed values and 20 duplicate retries. Compare pooled and route-specific summaries, retain outcome counts, and control how much detail reaches the final chart. The data is a constructed operational scenario, not a measurement of a real service.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through a misleading pooled average

Suppose fast requests improve from 80 to 60 ms and slow requests from 800 to 600 ms. Before the release, traffic is mostly fast; afterward it is mostly slow. The weighted overall mean can rise even when both conditional means fall. Calculate both levels before drawing a conclusion. Timeouts are absent durations here, not zero milliseconds; report them separately. A causal claim about the release would need a stronger comparison design.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(72);p2_records=[]
for p2_period,p2_route,p2_count,p2_mean in [('before','fast',900,80),('before','slow',100,800),('after','fast',100,60),('after','slow',900,600)]:
    for p2_value in p2_rng.lognormal(np.log(p2_mean)-0.5*0.15**2,0.15,p2_count):
        p2_records.append({'request_id':len(p2_records),'period':p2_period,'route':p2_route,'latency_ms':str(round(p2_value,2)),'status':'ok'})
p2_raw=pd.DataFrame(p2_records)
p2_raw.loc[::97,'latency_ms']='';p2_raw.loc[::97,'status']='timeout'
p2_raw.loc[[11,12,13],'latency_ms']='broken'
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-20:]],ignore_index=True)
p2_unique=p2_raw.drop_duplicates().copy()
p2_unique['latency']=pd.to_numeric(p2_unique.latency_ms,errors='coerce')
p2_bad=(p2_unique.status=='ok')&p2_unique.latency.isna()
p2_observed=p2_unique[(p2_unique.status=='ok')&~p2_bad].copy()
p2_summary=p2_observed.groupby(['period','route']).latency.agg(count='size',median='median',mean='mean',p95=lambda x:x.quantile(.95)).reset_index()
p2_means=p2_summary.pivot(index='route',columns='period',values='mean')
assert (p2_means['after']<p2_means['before']).all()
p2_pooled=p2_observed.groupby('period').latency.mean()
assert p2_pooled['after']>p2_pooled['before']
assert len(p2_unique)==2000 and p2_bad.sum()==3
print({'raw':len(p2_raw),'unique':len(p2_unique),'timeouts':int((p2_unique.status=='timeout').sum()),'malformed_ok_rows':int(p2_bad.sum())})
display(p2_summary)

### Work through detail versus payload size

Compare an all-event interactive scatter with a compact aggregate chart containing count and p95 in hover. Both are useful for different questions. The aggregate chart is smaller but cannot reveal every unusual request; the detailed chart should use a bounded sample or a server-backed drill-down for larger data.

Measure figure JSON separately from bundled JavaScript. A self-contained HTML file embeds the runtime and is intentionally larger; a CDN-based export is smaller but needs connectivity. We preserve a self-contained summary so it opens offline.

In [ ]:
import plotly.express as px
p2_detail=px.scatter(p2_observed,x='request_id',y='latency',color='route',facet_col='period',hover_data=['status'],render_mode='webgl')
p2_compact=px.bar(p2_summary,x='route',y='mean',color='period',barmode='group',
 hover_data={'count':True,'p95':':.1f','median':':.1f'},labels={'mean':'Mean observed success latency (ms)'},
 title='Route means; timeouts and malformed records excluded and counted separately')
p2_detail_bytes=len(p2_detail.to_json().encode());p2_summary_bytes=len(p2_compact.to_json().encode())
p2_compact.write_html(p2_out/'plotly_release_report.html',include_plotlyjs=True,full_html=True)
p2_summary.to_csv(p2_out/'plotly_release_summary.csv',index=False)
p2_compact.show()
print({'event_figure_json_bytes':p2_detail_bytes,'summary_figure_json_bytes':p2_summary_bytes,
 'excluded_timeout_rows':int((p2_unique.status=='timeout').sum()),'excluded_malformed_rows':int(p2_bad.sum())})
assert p2_summary_bytes<p2_detail_bytes
assert sum(len(trace.y) for trace in p2_compact.data)==len(p2_summary)

### Interpret the delivery constraint

The smaller figure answers a defined summary question and retains group sizes in hover. It is not a faster version of the same detailed view: it deliberately removes per-request exploration. If readers need debugging detail, provide a filtered drill-down with an explicit row budget and preserve source IDs.

**Try:** change a trace's visibility manually. **Worked reasoning:** titles and legends must still explain which period and population are shown. A dropdown that hides a trace without updating its explanatory text can produce a polished but misleading report. Inspect the exported HTML independently of the notebook.